In [5]:
import zipfile
unzip_files = zipfile.ZipFile('archive (1).zip','r')
unzip_files.extractall("Assignment_5")

In [9]:
import pandas as pd
import numpy as np
import re
import string
import random
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from nltk.corpus import stopwords
import gensim
from gensim.models import Word2Vec
import nltk
from nltk.tokenize import word_tokenize

In [8]:
df = pd.read_csv("C:/Users/nensi/SEM-6/ADML/Assignment_5/Reviews.csv")
df

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...
...,...,...,...,...,...,...,...,...,...,...
568449,568450,B001EO7N10,A28KG5XORO54AY,Lettie D. Carter,0,0,5,1299628800,Will not do without,Great for sesame chicken..this is a good if no...
568450,568451,B003S1WTCU,A3I8AFVPEE8KI5,R. Sawyer,0,0,2,1331251200,disappointed,I'm disappointed with the flavor. The chocolat...
568451,568452,B004I613EE,A121AA1GQV751Z,"pksd ""pk_007""",2,2,5,1329782400,Perfect for our maltipoo,"These stars are small, so you can give 10-15 o..."
568452,568453,B004I613EE,A3IBEVCTXKNOH,"Kathy A. Welch ""katwel""",1,1,5,1331596800,Favorite Training and reward treat,These are the BEST treats for training and rew...


In [10]:
df = df[['Text']].dropna().reset_index(drop=True)

In [11]:
nltk.download('stopwords')
nltk.download('punkt')
stop_words = set(stopwords.words('english'))
stop_words.discard('not')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nensi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\nensi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [12]:
def preprocess_text(text):
    text = text.lower()  
    text = re.sub(f'[{string.punctuation}]', '', text) 
    tokens = word_tokenize(text)  # Tokenize words
    tokens = [word for word in tokens if word not in stop_words] 
    return ' '.join(tokens)

In [13]:
df['Processed_Text'] = df['Text'].apply(preprocess_text)

In [14]:
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(df['Processed_Text'])

In [15]:
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(df['Processed_Text'])

In [16]:
sample_df = df.sample(n=2000, random_state=42)
sentences = [word_tokenize(text) for text in sample_df['Processed_Text']]

w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=2, workers=4)

word_vectors = {word: w2v_model.wv[word] for word in w2v_model.wv.index_to_key}

In [17]:
print("BoW shape:", X_bow.shape)
print("TF-IDF shape:", X_tfidf.shape)
print("Sample Word2Vec vector for 'good':", word_vectors.get('good', 'Not in vocab'))

BoW shape: (568454, 240245)
TF-IDF shape: (568454, 240245)
Sample Word2Vec vector for 'good': [-3.9427367e-01  5.5535930e-01  2.8417560e-01  1.3490114e-01
 -3.1259442e-03 -8.6517024e-01  1.3203610e-01  1.5162249e+00
 -5.2896726e-01 -1.6364245e-01 -4.1431454e-01 -9.1096669e-01
 -2.4398483e-01  2.1088509e-01  9.6784513e-03 -4.7511780e-01
  2.0729235e-01 -8.6039722e-01  1.6933623e-01 -1.0788751e+00
  6.3584334e-01  5.3622711e-01  1.2653767e-01 -1.3247572e-01
 -9.0608917e-02 -8.0859087e-02 -5.2970797e-01 -5.9311563e-01
 -5.4568601e-01  1.2196293e-01  6.2534028e-01  2.3432998e-01
  2.2894245e-01 -3.0391404e-01 -4.2497787e-01  9.0476847e-01
  3.9337862e-02 -4.7435054e-01 -2.7305421e-01 -1.2180146e+00
  1.9648032e-01 -7.2629488e-01 -2.1122314e-01 -5.7652581e-02
  3.9015630e-01 -5.8980644e-01 -6.2050414e-01 -7.0088506e-02
  1.0974460e-01  4.0754527e-01  2.8112206e-01 -6.6920447e-01
  3.0865418e-02 -1.7749271e-01 -5.8285809e-01  2.8739744e-01
  3.4990615e-01 -1.8158329e-01 -6.1036325e-01  3.494